# Video Translator — hosted deployment

Run the full Video Translator (FastAPI API + React UI) on a free GPU host and
open it in your browser. The heavy lifting — vocal separation, transcription,
translation, voice cloning, lip-sync — runs on the machine this notebook is
attached to, not on your laptop.

**Before you start:** this needs a **GPU runtime**.

In Colab: **Runtime → Change runtime type → T4 GPU** (or any GPU). Then
**Runtime → Run all**.


> ⏱️ First run takes 10–20 minutes: the package install pulls PyTorch and the ML
> stack. Later runs in the same session are fast because the install is cached.

## 0. Get the code (Colab)

Colab gives a GPU VM with a writable `/content`. We clone the repository there
and install FFmpeg, which the pipeline shells out to for every render.

In [ ]:
import os
import subprocess

REPO_DIR = "/content/video-translator"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Cloning the repository...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/gajjalaashok75-UI/video-translator", REPO_DIR], check=True)
else:
    print("Repository already present; updating...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

print("Installing FFmpeg...")
subprocess.run(["apt-get", "-qq", "update"], check=False)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
print("Ready:", REPO_DIR)

## 1. Check the machine

Confirms we have a GPU, picks the runtime directory for this platform, and makes
sure FFmpeg is present. Nothing here downloads models — that happens lazily on
the first render, so this cell is quick.

In [ ]:
import os
import sys

# The repository is cloned into the working directory below.
REPO_DIR = "/content/video-translator"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from src.deploy import runtime

platform = runtime.detect_platform()
report = runtime.hardware_report()

print(f"Platform:  {platform}")
print(f"Python:    {sys.version.split()[0]}")
if report["cuda_available"]:
    print(f"GPU:       {report['gpu_name']} ({report['vram_gb']} GB VRAM)")
else:
    print("GPU:       none — renders will run on CPU and will be slow")
print(f"Preset:    {report['recommended_preset']}")
for note in report["notes"]:
    print(f"  note: {note}")

if runtime.ensure_ffmpeg():
    print("FFmpeg:    ready")
else:
    print("FFmpeg:    NOT available — install it before rendering")

## 2. Point the runtime at a writable directory

The application writes temporary files, finished renders and job state. On a
hosted machine the checkout is not the right place for that (it is small, and on
Kaggle it is read-only). This cell redirects `TEMP_DIR`, `OUTPUT_DIR`,
`JOBS_DIR` and the uploads directory to a platform-appropriate location.

**It must run before the application is imported**, because `src.utils.config`
reads those variables at import time.

In [ ]:
from pathlib import Path

runtime_dirs = runtime.configure_runtime()
runtime_root = next(iter(runtime_dirs.values())).parent

print(f"Runtime root: {runtime_root}")
for name, path in runtime_dirs.items():
    print(f"  {name:8} -> {path}")

# Finished videos land in the output directory; remember the path for later.
OUTPUT_DIR = runtime_dirs["output"]

## 3. Install the application

Installs the package with the extras a GPU render needs: the CUDA PyTorch build,
the API layer, the default TTS and translation backends, audio processing, and
the optional stages the notebook advertises. This is the slow cell.

Every stage the UI can offer is installed here so the app reports it as
available rather than falling back: Whisper transcription (via
`translation-llm`), Google/NLLB translation, Edge TTS, Demucs vocal
separation, speaker diarization, lip-sync and on-screen text translation.
Diarization and IndicF5 voice cloning need a Hugging Face token (see step 5);
the rest runs with no keys.

In [ ]:
import subprocess
import sys

PIP_EXTRAS = "[gpu-cu124,api,tts,translation-llm,audio-processing,diarization,lipsync,ocr]"

# `-e` so the repo is importable as `src.*` and edits are picked up.
cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "-e", f"{REPO_DIR}{PIP_EXTRAS}",
]
print("Installing:", " ".join(cmd))
result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    raise SystemExit(
        "The install failed. Re-run this cell after reading the error above; "
        "a transient download failure usually succeeds on a retry.")
print("Install complete.")

## 4. Start the server

Starts the API and the built UI in this process and waits until it answers
`/api/health`. The cell prints the URL to open.

The server runs in a background thread of this kernel, so the notebook stays
responsive. Re-running the cell stops the previous server first, so you never
end up with two processes fighting over the port.

In [ ]:
# Stop any server a previous run of this cell left behind.
_previous = globals().get("server")
if _previous is not None:
    _previous.stop()
    print("Stopped the previous server.")

PORT = 8000
server = runtime.start_server(host="0.0.0.0", port=PORT, ready_timeout=300)

url = runtime.notebook_public_url(PORT)
print()
print("=" * 62)
print("  Video Translator is running")
print(f"  Open:  {url}")
print("=" * 62)
print()
print("Keep this notebook tab open while you use the app.")

# On Colab this URL is proxied automatically and needs no account or token.
# It is valid only while this notebook is running.

## 5. Use it

1. Open the URL printed above.
2. On the **Dub** page, drop in a video and pick the source and target
   languages. Hindi and the other Indic languages are supported.
3. Leave the quality preset at the recommended value shown in step 1
   (`FAST` on a CPU box, `BALANCED` or `HIGH` on a GPU).
4. Press **Start** and watch the progress.

Renders are written to the output directory shown in step 2. The **Jobs** page
lists everything this session has produced.

### Optional: a Hugging Face token

Speaker diarization, IndicF5 voice cloning (the 11 Indian languages) and the
Llama/ALMA translation models are gated on Hugging Face. Set it before
starting the server, then restart the server cell:

```python
import os
os.environ["HF_TOKEN"] = "hf_..."   # or use the Secrets panel on the left
```

Without it those stages report as unavailable and the pipeline falls back:
diarization runs as a single speaker, Indian languages speak via Edge TTS
instead of a cloned IndicF5 voice, and Llama/ALMA translation is not offered.
Lip-sync (Wav2Lip/LivePortrait) and on-screen text translation (PaddleOCR) are
installed by this notebook and work with no token.

## 6. Stop the server

Frees the port and the GPU memory. Run this when you are done, or just close the
notebook — the server is a daemon thread and dies with the kernel.

In [ ]:
server.stop()
print("Server stopped:", not server.is_running())

## Troubleshooting

| Symptom | Fix |
|---|---|
| `The server did not become ready` | Re-run the install cell; a partial install is the usual cause. Check the log output above the error. |
| Out of GPU memory during a render | Use the `FAST` preset (no lip-sync), or process a shorter clip. |
| `ffmpeg not found` | Re-run step 1; on Colab and Kaggle the cell installs it. |
| The URL does not open | The link is only reachable from inside the VM. Re-run step 4 to get a fresh proxy URL, and keep the notebook open. |
| Diarization, Llama translation or IndicF5 cloning unavailable | Set `HF_TOKEN` (see step 5) and restart the server. |